In [ ]:
import os

#API Token
os.environ['KAGGLE_USERNAME'] = "rpranit01"
os.environ['KAGGLE_KEY'] = "KGAT_1fbb520ed76c5f8773ef758c86a7920f"

!kaggle datasets list

ref                                                               title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
rauffauzanrambe/fifa-world-cup-2026-player-performance-dataset    FIFA World Cup 2026 Player Performance Dataset       4154062  2026-06-10 12:58:47.093000          17980        437                1  
uditjain13/credit-card-fraud-detection-2026                       Credit Card Fraud Detection 2026                      665173  2026-07-16 19:17:54.650000           1025         25                1  
abbas829/ecommerce-sales-dataset                                  Ecommerce sales dataset                               110951  2026-07-03 00:59:03.347000           4410         74                1  


In [ ]:
!pip install --upgrade kaggle

In [ ]:
#Installing kagglehub
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("arbethi/diabetic-retinopathy-level-detection")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'diabetic-retinopathy-level-detection' dataset.
Path to dataset files: /kaggle/input/diabetic-retinopathy-level-detection


In [ ]:
#training and testing directory locations
train_path = os.path.join(path, 'train')
test_path = os.path.join(path, 'test')

if os.path.exists(train_path):
    print("Training subfolders (Classes):", os.listdir(train_path))
    print("Testing subfolders:", os.listdir(test_path))
else:
    print("Directory root subfolders:", os.listdir(path))

Directory root subfolders: ['preprocessed dataset', 'inception-diabetic.h5']


In [ ]:
dataset_root = os.path.join(path, 'preprocessed dataset')
print("Contents of preprocessed folder:", os.listdir(dataset_root))

Contents of preprocessed folder: ['preprocessed dataset']


In [ ]:
deep_root = os.path.join(path, 'preprocessed dataset', 'preprocessed dataset')
print("Contents of deeper folder:", os.listdir(deep_root))

Contents of deeper folder: ['training', 'testing']


In [ ]:
#final location
train_path = os.path.join(path, 'preprocessed dataset', 'preprocessed dataset', 'training')
test_path = os.path.join(path, 'preprocessed dataset', 'preprocessed dataset', 'testing')
print("Training classes found:", os.listdir(train_path))
print("Testing classes found:", os.listdir(test_path))

Training classes found: ['2', '0', '3', '1', '4']
Testing classes found: ['2', '0', '3', '1', '4']


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#train generator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True
)

#test generator
test_datagen = ImageDataGenerator(rescale=1./255)

BATCH_SIZE = 64
TARGET_SIZE = (299, 299)

print("Loading Training Images...")
train_set = train_datagen.flow_from_directory(
    directory=train_path,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("\nLoading Testing Images...")
test_set = test_datagen.flow_from_directory(
    directory=test_path,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Loading Training Images...
Found 3662 images belonging to 5 classes.

Loading Testing Images...
Found 734 images belonging to 5 classes.


In [ ]:
#importing libraries

from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
#init model
base_model = Xception(
    input_shape=(299, 299, 3),
    weights='imagenet',
    include_top=False
)

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
x = Flatten()(base_model.output)
predictions = Dense(5, activation='softmax')(x) #5 neurons since there are 5 classes

In [ ]:
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 149, 149,  │        864 │ input_layer_3[0]… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 149, 149,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 149, 149,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 147, 147,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 147, 147,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 147, 147,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 147, 147,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 147, 147,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 147, 147,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 147, 147,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 147, 147,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 74, 74,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 74, 74,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 74, 74,    │        512 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_36 (Add)        │ (None, 74, 74,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 74, 74,    │          0 │ add_36[0][0]    

 Total params: 21,885,485 (83.49 MB)

 Trainable params: 1,024,005 (3.91 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [ ]:
checkpoint = ModelCheckpoint(
    filepath='diabetic_retinopathy_xception.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
print("Starting training pipeline for 10 epochs...")
history = model.fit(
    train_set,
    epochs=10,
    validation_data=test_set,
    callbacks=[checkpoint]
)

Starting training pipeline for 10 epochs...
Epoch 1/10
 2/58 ━━━━━━━━━━━━━━━━━━━━ 29s 535ms/step - accuracy: 0.1953 - loss: 4.4661

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

In [ ]:
#accuracy
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', marker='o')
plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='x')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

#loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', marker='o')
plt.plot(epochs_range, val_loss, label='Validation Loss', marker='x')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (Categorical Crossentropy)')

plt.show()

In [ ]:
from tensorflow.keras.models import load_model

print("Loading the optimized weights checkpoint...")
saved_model = load_model('diabetic_retinopathy_xception.h5')

print("Evaluating performance across the complete testing dataset...")
test_loss, test_accuracy = saved_model.evaluate(test_set, verbose=1)

print("\n==== FINAL PERFORMANCE METRICS ====")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
!pip install ibm_cloudant

In [ ]:
from ibm_cloudant.client import Cloudant

CLOUDANT_USERNAME = "f8258608-6c52-45dd-b8b0-e3680202a56a-bluemix"
CLOUDANT_PASSWORD = "IHxulWxvSy_TNtIjAqihkadCZ9xAfSO7ftC8Zw3VnJ6I"
CLOUDANT_URL = "https://f8258608-6c52-45dd-b8b0-e3680202a56a-bluemix.cloudantnosqldb.appdomain.cloud"
DB_NAME = "diabetic-retinopathy-db"

try:
    client = Cloudant(CLOUDANT_USERNAME, CLOUDANT_PASSWORD, url=CLOUDANT_URL, connect=True)

    if DB_NAME in client.all_dbs():
        db = client[DB_NAME]
        print(f"Successfully connected to '{DB_NAME}'!")
    else:
        db = client.create_database(DB_NAME)
        print(f"Created and connected to '{DB_NAME}'!")

except Exception as e:
    print(f"Connection failed: {e}")